In [ ]:
from google import genai
from google.genai import types

In [ ]:
sample_data = [
    {"month": "Jan", "product": "Insight Basic", "revenue": 12400, "users": 320},
    {"month": "Feb", "product": "Insight Basic", "revenue": 15100, "users": 380},
    {"month": "Mar", "product": "Insight Basic", "revenue": 16800, "users": 410},
    {"month": "Jan", "product": "Insight Pro", "revenue": 18900, "users": 210},
    {"month": "Feb", "product": "Insight Pro", "revenue": 22400, "users": 260},
    {"month": "Mar", "product": "Insight Pro", "revenue": 24700, "users": 295},
]

user_query = "Show how monthly revenue changes for each product."
chosen_visualization = "line chart"

In [ ]:
def build_visual(query, data, visualization, api_key, model):
    """Ask Gemini to generate Python code for a user-selected data visualization."""
    client = genai.Client(api_key=api_key)

    try:
        generation = client.models.generate_content(
        model=model,
        
        config=types.GenerateContentConfig(
            system_instruction="You are a Python data-visualization assistant. Return only executable Python code, with no markdown fences or explanatory prose.",
            temperature=0.1),
        
        contents=f"""
                Create Python code that visualizes the provided data according to the user's request.

                User request:
                {query}

                Visualization chosen by the user:
                {visualization}

                Data:
                {data}

                Requirements:
                - Return only Python code.
                - Use pandas to load the data into a DataFrame.
                - Use matplotlib or seaborn for the chart.
                - Create the visualization type chosen by the user.
                - Include a clear title and axis labels.
                - Keep the code concise and runnable in a notebook cell.
                - Add the necessary imports for the code to run successfully.
                """
    )
        
        code_parts = []
        
        if hasattr(generation, "candidates"):
            for cand in generation.candidates:
                if hasattr(cand, "content") and cand.content and hasattr(cand.content, "parts"):
                    for part in cand.content.parts:
                        if hasattr(part, "text") and part.text:
                            code_parts.append(part.text)
                            
        response = "\n".join(code_parts).strip()
                            
        return response
        
    except Exception as e:
        print(f"Error generating content: {e}")
        return None

In [ ]:
api_key = "YOUR_GEMINI_API_KEY"
model = "gemini-2.0-flash"

generated_code = build_visual(
    query=user_query,
    data=sample_data,
    visualization=chosen_visualization,
    api_key=api_key,
    model=model,
)

print(generated_code)

if generated_code:
    exec(generated_code)